# Generate Synthetic CS2 Commentary Training Data

Use this notebook to generate synthetic training rows for a Counter-Strike 2 commentary model.

The notebook expects seed examples shaped like the current v3 wrapper output:

```json
{
  "input": {
    "context": {
      "score": {...},
      "alive_players": [...]
    },
    "previous_events": [...],
    "current_events": [...],
    "request": {
      "mode": "event_bundle",
      "lines": [...]
    }
  }
}
```

It generates synthetic rows shaped like:

```json
{
  "input": {...},
  "output": {
    "commentary": "line 1\nline 2"
  }
}
```

This notebook is specifically for **CS2**. The only caster labels used are `play_by_play` and `color`.


## Data generation settings


In [1]:
NUM_TRAIN_EXAMPLES = 5  # @param {type:"number"}
NUM_VAL_EXAMPLES = 1  # @param {type:"number"}
NUM_TEST_EXAMPLES = 1  # @param {type:"number"}
TEMPERATURE = 0.9  # @param {type:"number"}
MAX_ROW_RETRIES = 5  # @param {type:"number"}
FREQUENCY_PENALTY = 0.5  # @param {type:"number"}
PRESENCE_PENALTY = 0.15  # @param {type:"number"}

DATA_FOLDER = "./.data/generated_cs2"
SEED_EXAMPLES_FILE = "./training_wrapper_pretty.jsonl"  # upload or mount this in Colab
MAX_SEED_EXAMPLES = 50  # @param {type:"number"}
SEED_EXAMPLES_PER_PROMPT = 3  # @param {type:"number"}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.4-nano"

ALLOWED_CASTERS = ["play_by_play", "color"]
ALLOWED_PROMPT_STYLES = ["play_by_play_event", "play_by_play_follow_up", "idle_color"]
ALLOWED_REQUEST_MODES = ["event_bundle", "idle_color", "idle_conversation"]

INLINE_SEED_EXAMPLES = []

from pathlib import Path
Path(DATA_FOLDER).mkdir(parents=True, exist_ok=True)


## Seed examples and schema


In [2]:
import json
from pathlib import Path

def load_seed_examples(seed_file: str, inline_examples: list[dict], max_examples: int) -> list[dict]:
    examples: list[dict] = []
    path = Path(seed_file)

    if path.exists():
        text = path.read_text(encoding="utf-8")

        chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
        if chunks:
            for chunk in chunks:
                record = json.loads(chunk)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)
        else:
            for raw_line in text.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                record = json.loads(line)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)

    for record in inline_examples:
        if isinstance(record, dict) and isinstance(record.get("input"), dict):
            examples.append(record)

    if max_examples > 0:
        examples = examples[:max_examples]

    return examples


SEED_EXAMPLES = load_seed_examples(SEED_EXAMPLES_FILE, INLINE_SEED_EXAMPLES, MAX_SEED_EXAMPLES)
print(f"Loaded {len(SEED_EXAMPLES)} seed examples")
if SEED_EXAMPLES:
    print(json.dumps(SEED_EXAMPLES[0], indent=2, sort_keys=True))


## Model for structured output


In [3]:
from typing import Any, Literal
from pydantic import BaseModel, Field, field_validator, model_validator


class RequestLine(BaseModel):
    caster: Literal["play_by_play", "color"]
    style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"]


class RequestSpec(BaseModel):
    mode: Literal["event_bundle", "idle_color", "idle_conversation"]
    lines: list[RequestLine] = Field(default_factory=list)


class SyntheticInput(BaseModel):
    context: dict[str, Any]
    previous_events: list[dict[str, Any]] = Field(default_factory=list)
    current_events: list[dict[str, Any]] = Field(default_factory=list)
    request: RequestSpec


class OutputLine(BaseModel):
    caster: Literal["play_by_play", "color"]
    text: str

    @field_validator("text")
    @classmethod
    def validate_text(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("output line text must not be empty")
        return value


class SyntheticOutput(BaseModel):
    lines: list[OutputLine] = Field(default_factory=list)


class SyntheticTrainingRow(BaseModel):
    input: SyntheticInput
    output: SyntheticOutput

    @model_validator(mode="after")
    def validate_alignment(self):
        request_lines = self.input.request.lines
        output_lines = self.output.lines

        if len(request_lines) != len(output_lines):
            raise ValueError(
                f"output.lines must contain exactly {len(request_lines)} entries, got {len(output_lines)}"
            )

        for idx, (request_line, output_line) in enumerate(zip(request_lines, output_lines), start=1):
            if request_line.caster != output_line.caster:
                raise ValueError(
                    f"output line {idx} caster must be {request_line.caster}, got {output_line.caster}"
                )

            max_words = 12 if output_line.caster == "play_by_play" else 28
            word_count = len(output_line.text.split())
            if word_count > max_words:
                raise ValueError(
                    f"output line {idx} exceeds the {max_words}-word limit with {word_count} words"
                )

        return self


## Get OpenRouter API key


In [4]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Synthetic generation functions


In [5]:
import json
import openai
import os
import random

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


COMMENTARY_SYSTEM_PROMPT = """
You generate structured commentary for a Counter-Strike 2 casting system.

STRICT REQUIREMENTS:
- Output valid JSON only.
- Follow the response schema exactly.
- No markdown, no code fences, no explanations.
- Keep every line speakable for TTS.
- Avoid repeating the same verbs, phrases, or sentence shapes across lines.

CASTERS:

1. play_by_play = Portal 2 Announcer inspired.
- Cold, precise, clinical, authoritative.
- Short, sharp, controlled.
- Focus on what just happened or the most important current state.
- Can sound ominous or procedural.
- Never mention Portal, Aperture, Announcer, or any copyrighted title directly.

2. color = Portal 2 Turret inspired.
- Quirky, polite, slightly unstable, eager.
- Can react, joke, expand, or lightly banter.
- Stronger personality than play_by_play.
- May include nuanced CS2 insights, positioning reads, timing comments, or tactical context.
- Can gently respond to what the other caster just said.
- Never mention Portal, Turret, Aperture, or any copyrighted title directly.

MODE RULES:

A. event_bundle
- Usually the first line is direct play-by-play on the trigger event.
- Follow-up lines expand on positioning, consequences, tactical meaning, momentum, or character reaction.
- If game_over is present, acknowledge the result clearly.
- If both a kill and game_over are present, line 1 can mention the kill and line 2 can land the result.

B. idle_color
- These lines are freer and more personality-driven.
- Even when three lines are requested, they may feel like one continuing thought broken into short spoken beats.
- Lean into map state, pacing, setup, score pressure, player positions, CS2 knowledge, or light in-universe flavor.

C. idle_conversation
- Treat it like brief caster chemistry.
- Lines may bounce off each other with retorts, questions, dry replies, inside jokes, or contrasting reads.
- Still keep the content grounded in the current game state.

CONTENT RULES:
- Prefer exact player names and map callouts when available.
- Use round_kills, alive player counts, score, and map positions when they create better commentary.
- If the situation is tactical, make it sound like real high-level CS2 commentary.
- If a line can be more specific, make it more specific.
- Avoid generic filler like: huge kill, big round, what a play, comes up massive, nice shot, locks it down.
- Avoid repetitive openers and repetitive sentence templates.
- Do not invent unsupported facts.
- Do not mention being an AI, model, or generator.

LINE LENGTH:
- play_by_play lines: ideally 4 to 10 words, hard max 12 words.
- color lines: ideally 6 to 20 words, hard max 28 words.
- color text may contain 1 to 3 short sentences inside the same string.
""".strip()


FEW_SHOT_EXAMPLES = [
    {
        "input": {
            "context": {
                "score": {"CT": 2, "T": 5},
                "alive_players": [
                    {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    {"name": "Maru", "team": "T", "map_callout": "B Site"}
                ]
            },
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    "victim": {"name": "Maru", "team": "T"}
                }
            ],
            "request": {
                "mode": "event_bundle",
                "lines": [
                    {"caster": "play_by_play", "style": "play_by_play_event"},
                    {"caster": "color", "style": "play_by_play_follow_up"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "play_by_play", "text": "Colin deletes Maru at B Doors."},
                {"caster": "color", "text": "That angle was trapped. B Site had nowhere soft to breathe."}
            ]
        }
    },
    {
        "input": {
            "context": {
                "score": {"CT": 4, "T": 4},
                "alive_players": [
                    {"name": "Niles", "team": "T", "map_callout": "Top Mid"}
                ]
            },
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "grenade_detonated",
                    "grenade_type": "smoke",
                    "detonation_callout": "Top Mid",
                    "owner_player": {"name": "Niles", "team": "T", "map_callout": "Top Mid"}
                }
            ],
            "request": {
                "mode": "event_bundle",
                "lines": [
                    {"caster": "play_by_play", "style": "play_by_play_event"},
                    {"caster": "color", "style": "play_by_play_follow_up"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "play_by_play", "text": "Smoke deployed at Top Mid."},
                {"caster": "color", "text": "Mid vision disappears. Very tidy. Very inconvenient for the peekers."}
            ]
        }
    },
    {
        "input": {
            "context": {
                "score": {"CT": 6, "T": 5},
                "alive_players": [
                    {"name": "Yanni", "team": "CT", "map_callout": "A Site"},
                    {"name": "Tony", "team": "T", "map_callout": "Long"}
                ]
            },
            "previous_events": [
                {"event_type": "bomb_event", "state_after": "planted"}
            ],
            "current_events": [],
            "request": {
                "mode": "idle_color",
                "lines": [
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "color", "text": "This hold is getting narrow."},
                {"caster": "color", "text": "Long pressure keeps A very honest."},
                {"caster": "color", "text": "One mistimed swing and everything becomes confetti."}
            ]
        }
    },
    {
        "input": {
            "context": {
                "score": {"CT": 7, "T": 6},
                "alive_players": [
                    {"name": "Walt", "team": "CT", "map_callout": "Top Mid"},
                    {"name": "Uri", "team": "T", "map_callout": "Xbox"}
                ]
            },
            "previous_events": [
                {"event_type": "grenade_detonated", "grenade_type": "smoke", "detonation_callout": "Xbox"}
            ],
            "current_events": [],
            "request": {
                "mode": "idle_conversation",
                "lines": [
                    {"caster": "play_by_play", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "play_by_play", "style": "idle_color"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "play_by_play", "text": "Mid remains under CT surveillance."},
                {"caster": "color", "text": "Yes. Very watched. I would not frolic through that smoke."},
                {"caster": "play_by_play", "text": "Wise. The punishment would be immediate."}
            ]
        }
    }
]


RESPONSE_JSON_SCHEMA = {
    "name": "synthetic_training_row",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "input": {
                "type": "object",
                "additionalProperties": True,
                "properties": {
                    "context": {"type": "object", "additionalProperties": True},
                    "previous_events": {
                        "type": "array",
                        "items": {"type": "object", "additionalProperties": True}
                    },
                    "current_events": {
                        "type": "array",
                        "items": {"type": "object", "additionalProperties": True}
                    },
                    "request": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "mode": {"type": "string", "enum": ["event_bundle", "idle_color", "idle_conversation"]},
                            "lines": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "additionalProperties": False,
                                    "properties": {
                                        "caster": {"type": "string", "enum": ["play_by_play", "color"]},
                                        "style": {"type": "string", "enum": ["play_by_play_event", "play_by_play_follow_up", "idle_color"]}
                                    },
                                    "required": ["caster", "style"]
                                }
                            }
                        },
                        "required": ["mode", "lines"]
                    }
                },
                "required": ["context", "previous_events", "current_events", "request"]
            },
            "output": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "lines": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "properties": {
                                "caster": {"type": "string", "enum": ["play_by_play", "color"]},
                                "text": {"type": "string"}
                            },
                            "required": ["caster", "text"]
                        }
                    }
                },
                "required": ["lines"]
            }
        },
        "required": ["input", "output"]
    }
}


def build_seed_block(seed_examples: list[dict]) -> str:
    if not seed_examples:
        return "[]"
    return json.dumps(seed_examples, indent=2, sort_keys=True)


def build_few_shot_block() -> str:
    return json.dumps(FEW_SHOT_EXAMPLES, indent=2, sort_keys=True)


def choose_request() -> dict:
    mode = random.choice(ALLOWED_REQUEST_MODES)
    if mode == "event_bundle":
        return {
            "mode": "event_bundle",
            "lines": [
                {"caster": "play_by_play", "style": "play_by_play_event"},
                {"caster": "color", "style": "play_by_play_follow_up"}
            ],
        }
    if mode == "idle_conversation":
        return {
            "mode": "idle_conversation",
            "lines": [
                {"caster": "play_by_play", "style": "idle_color"},
                {"caster": "color", "style": "idle_color"},
                {"caster": "play_by_play", "style": "idle_color"}
            ],
        }
    return {
        "mode": "idle_color",
        "lines": [
            {"caster": "color", "style": "idle_color"},
            {"caster": "color", "style": "idle_color"},
            {"caster": "color", "style": "idle_color"}
        ],
    }


def extract_json_object(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]
        raw_text = raw_text.strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"Model did not return a valid JSON object: {raw_text[:400]}")
        candidate = raw_text[start:end+1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError as error:
            raise ValueError(f"Model returned invalid JSON: {candidate[:400]}") from error


def build_generation_prompt(request: dict, sampled_examples: list[dict]) -> str:
    expected_line_count = len(request["lines"])
    return f"""
Generate exactly one new synthetic training row.

The generated row must use this exact request object inside input.request:
{json.dumps(request, indent=2, sort_keys=True)}

Hard requirements:
- input.request must match the exact object above.
- output.lines must contain exactly {expected_line_count} entries.
- Each output line caster must match the corresponding input.request.lines caster.
- Keep the content grounded in current_events when current_events is non-empty.
- previous_events is context only and must not override current_events.
- context.score and context.alive_players are the only compact global context.
- Prefer exact player names, score pressure, alive counts, and map callouts when useful.
- For event_bundle, make the follow-up line richer and more specific than a generic reaction.
- For idle_color and idle_conversation, you can be freer, more characterful, and more analytical.
- Color commentary may contain 1 to 3 short sentences inside one text string.
- Avoid repeated phrase openings and repeated generic esports filler.

Few-shot examples:
{build_few_shot_block()}

Seed examples for structure and event diversity:
{build_seed_block(sampled_examples)}
""".strip()


def call_structured_model(system_prompt: str, user_prompt: str) -> dict:
    completion = client.chat.completions.create(
        model=DATAGEN_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_schema", "json_schema": RESPONSE_JSON_SCHEMA},
        temperature=TEMPERATURE,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
    )

    message = completion.choices[0].message
    content = message.content or ""
    if not content:
        raise ValueError("Model returned empty content")
    return extract_json_object(content)


def validate_generated_row(parsed: dict, expected_request: dict) -> SyntheticTrainingRow:
    row = SyntheticTrainingRow.model_validate(parsed)

    actual_request = row.input.request.model_dump()
    if actual_request != expected_request:
        raise ValueError(
            "Generated input.request did not match the expected request. "
            f"Expected {expected_request}, got {actual_request}"
        )

    for output_line in row.output.lines:
        if "\n" in output_line.text:
            raise ValueError("Each output line must be a single string without embedded newlines")

    return row


def generate_synthetic_row(seed_examples: list[dict]) -> SyntheticTrainingRow:
    if not seed_examples:
        raise ValueError("No seed examples loaded. Upload or point SEED_EXAMPLES_FILE to the training wrapper JSONL.")

    sampled_examples = random.sample(seed_examples, k=min(SEED_EXAMPLES_PER_PROMPT, len(seed_examples)))
    request = choose_request()
    prompt = build_generation_prompt(request=request, sampled_examples=sampled_examples)
    parsed = call_structured_model(COMMENTARY_SYSTEM_PROMPT, prompt)
    return validate_generated_row(parsed, expected_request=request)



## Dataset generation functions


In [6]:
from tqdm import tqdm


def generate_dataset(num_examples: int, filename: str) -> None:
    if "SEED_EXAMPLES" not in globals():
        raise RuntimeError("SEED_EXAMPLES is not defined. Run the seed-loading cell first.")
    if not SEED_EXAMPLES:
        raise RuntimeError("SEED_EXAMPLES is empty. Check SEED_EXAMPLES_FILE and reload seeds.")

    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):
            row = None
            last_error = None
            for attempt in range(1, MAX_ROW_RETRIES + 1):
                try:
                    row = generate_synthetic_row(SEED_EXAMPLES)
                    break
                except Exception as error:
                    last_error = error
                    print(f"Error generating row {idx} on attempt {attempt}/{MAX_ROW_RETRIES}: {error}")
            if row is None:
                raise RuntimeError(f"Failed to generate row {idx} after {MAX_ROW_RETRIES} attempts") from last_error
            f.write(row.model_dump_json() + "\n")
            f.flush()



## Generate all the data!


In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y-%m-%d-%H:%M:%S')
TRAIN_FILE = f"{DATA_FOLDER}/train_{timestamp}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{timestamp}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{timestamp}.jsonl"

GENERATED_FILES = []

if NUM_TRAIN_EXAMPLES > 0:
    generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
    GENERATED_FILES.append(("train", TRAIN_FILE))

if NUM_VAL_EXAMPLES > 0:
    generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
    GENERATED_FILES.append(("valid", VALID_FILE))

if NUM_TEST_EXAMPLES > 0:
    generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)
    GENERATED_FILES.append(("test", TEST_FILE))

for split_name, split_path in GENERATED_FILES:
    print(f"{split_name}: {split_path}")


In [ ]:
# Preview generated rows
import json
from pathlib import Path

for split_name, split_path in GENERATED_FILES:
    preview_path = Path(split_path)
    if not preview_path.exists():
        continue
    print(f"=== {split_name} preview ===")
    for raw_line in preview_path.read_text(encoding='utf-8').splitlines()[:3]:
        row = json.loads(raw_line)
        print(json.dumps(row, indent=2, sort_keys=True))
        print("flattened_lines:")
        for line in row.get("output", {}).get("lines", []):
            print(f"  [{line['caster']}] {line['text']}")
        print('---')
